In [1]:
import os

os.environ["JAVA_HOME"] = r"C:\Program Files\Microsoft\jdk-17.0.20.101-hotspot"

os.environ["HADOOP_HOME"] = r"C:\hadoop"

os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ["PATH"]

print("JAVA_HOME =", os.environ["JAVA_HOME"])
print("HADOOP_HOME =", os.environ["HADOOP_HOME"])
print(
    "winutils exists =",
    os.path.exists(r"C:\hadoop\bin\winutils.exe"),
)
print(
    "hadoop.dll exists =",
    os.path.exists(r"C:\hadoop\bin\hadoop.dll"),
)

JAVA_HOME = C:\Program Files\Microsoft\jdk-17.0.20.101-hotspot
HADOOP_HOME = C:\hadoop
winutils exists = True
hadoop.dll exists = True


In [2]:
from __future__ import annotations

import json
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from credit_risk.utils.spark import create_spark_session
from xgboost.spark import SparkXGBClassifierModel

from credit_risk.modelling.artifacts_spark import (
    load_spark_model_artifacts,
)
from credit_risk.utils.config import read_config

In [3]:
from pathlib import Path
import os

if "project_path" not in globals():
    project_path = Path.cwd().parent
    os.chdir(project_path)

print("Project path:", project_path)

Project path: c:\Users\vorad\OneDrive\Desktop\Projects\mortgage-credit-risk


In [4]:
config = read_config(project_path)

In [5]:
spark = create_spark_session(config)

c:\Users\vorad\OneDrive\Desktop\Projects\mortgage-credit-risk\venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [6]:
print(
    "Algorithm:",
    config["parameters"]["modelling"]["algorithm"],
)

print(
    "Engine:",
    config["parameters"]["engine"],
)

Algorithm: xgboost
Engine: pyspark


In [7]:
features_config = config["parameters"]["modelling"]["features"]

numerical_features = features_config.get(
    "numerical_features",
    [],
)

categorical_features = features_config.get(
    "categorical_features",
    [],
)

engineered_features = features_config.get(
    "engineered_features",
    [],
)

configured_features = numerical_features + categorical_features + engineered_features

print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))
print("Engineered features:", len(engineered_features))
print("Total configured features:", len(configured_features))

Numerical features: 16
Categorical features: 8
Engineered features: 1
Total configured features: 25


In [8]:
for i, feature in enumerate(configured_features):
    print(f"{i:3d}  {feature}")

  0  number_of_borrowers
  1  mi_percentage
  2  original_upb
  3  original_interest_rate
  4  original_loan_term
  5  current_actual_upb
  6  current_interest_rate
  7  estimated_ltv
  8  calculated_loan_age
  9  remaining_months_to_legal_maturity
 10  current_dpd_numeric
 11  max_dpd_to_date
 12  delinquency_months_to_date
 13  months_since_last_delinquency
 14  upb_change_from_origination
 15  upb_pct_change_from_origination
 16  first_time_homebuyer_flag
 17  property_type
 18  occupancy_status
 19  loan_purpose
 20  channel
 21  super_conforming_flag
 22  harp_indicator
 23  property_state
 24  original_dti_missing


In [9]:
model, preprocessor = load_spark_model_artifacts(config)

In [11]:
xgb_model = model

importance = xgb_model.get_booster().get_score(importance_type="gain")

importance_df = (
    pd.DataFrame(
        importance.items(),
        columns=["feature", "importance"],
    )
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

importance_df["importance_pct"] = (
    importance_df["importance"] / importance_df["importance"].sum() * 100
)

importance_df.head(20)

,feature,importance,importance_pct
0,f10,3692.408447,26.866874
1,f11,2280.193604,16.591251
2,f6,1646.572632,11.980868
3,f8,844.274414,6.143149
4,f47,493.572784,3.591357
5,f3,479.788757,3.491061
6,f26,332.900726,2.422268
7,f17,296.057343,2.154186
8,f0,267.407776,1.945725
9,f27,263.220490,1.915257


In [12]:
for i, stage in enumerate(preprocessor.stages):
    print(f"\n{i}: {type(stage).__name__}")

    for param, value in stage.extractParamMap().items():
        name = param.name

        if name in ["inputCol", "inputCols", "outputCol", "outputCols"]:
            print(f"  {name}: {value}")


0: ImputerModel
  outputCol: ImputerModel_a72d77abd85c__output
  inputCols: ['number_of_borrowers', 'mi_percentage', 'original_upb', 'original_interest_rate', 'original_loan_term', 'current_actual_upb', 'current_interest_rate', 'estimated_ltv', 'calculated_loan_age', 'remaining_months_to_legal_maturity', 'current_dpd_numeric', 'max_dpd_to_date', 'delinquency_months_to_date', 'months_since_last_delinquency', 'upb_change_from_origination', 'upb_pct_change_from_origination']
  outputCols: ['__imputed_number_of_borrowers', '__imputed_mi_percentage', '__imputed_original_upb', '__imputed_original_interest_rate', '__imputed_original_loan_term', '__imputed_current_actual_upb', '__imputed_current_interest_rate', '__imputed_estimated_ltv', '__imputed_calculated_loan_age', '__imputed_remaining_months_to_legal_maturity', '__imputed_current_dpd_numeric', '__imputed_max_dpd_to_date', '__imputed_delinquency_months_to_date', '__imputed_months_since_last_delinquency', '__imputed_upb_change_from_origin

In [13]:
encoder = [s for s in preprocessor.stages if type(s).__name__ == "OneHotEncoderModel"][
    0
]

for inp, out in zip(encoder.getInputCols(), encoder.getOutputCols()):
    print(inp, "->", out)

__indexed_first_time_homebuyer_flag -> __encoded_first_time_homebuyer_flag
__indexed_property_type -> __encoded_property_type
__indexed_occupancy_status -> __encoded_occupancy_status
__indexed_loan_purpose -> __encoded_loan_purpose
__indexed_channel -> __encoded_channel
__indexed_super_conforming_flag -> __encoded_super_conforming_flag
__indexed_harp_indicator -> __encoded_harp_indicator
__indexed_property_state -> __encoded_property_state


In [14]:
assembler = [s for s in preprocessor.stages if type(s).__name__ == "VectorAssembler"][0]

print(assembler.getInputCols())

['__imputed_number_of_borrowers', '__imputed_mi_percentage', '__imputed_original_upb', '__imputed_original_interest_rate', '__imputed_original_loan_term', '__imputed_current_actual_upb', '__imputed_current_interest_rate', '__imputed_estimated_ltv', '__imputed_calculated_loan_age', '__imputed_remaining_months_to_legal_maturity', '__imputed_current_dpd_numeric', '__imputed_max_dpd_to_date', '__imputed_delinquency_months_to_date', '__imputed_months_since_last_delinquency', '__imputed_upb_change_from_origination', '__imputed_upb_pct_change_from_origination', '__encoded_first_time_homebuyer_flag', '__encoded_property_type', '__encoded_occupancy_status', '__encoded_loan_purpose', '__encoded_channel', '__encoded_super_conforming_flag', '__encoded_harp_indicator', '__encoded_property_state', 'original_dti_missing']


In [15]:
encoder = [s for s in preprocessor.stages if type(s).__name__ == "OneHotEncoderModel"][
    0
]

print("Input columns:")
print(encoder.getInputCols())

print("\nOutput columns:")
print(encoder.getOutputCols())

print("\nCategory sizes:")
print(encoder.categorySizes)

Input columns:
['__indexed_first_time_homebuyer_flag', '__indexed_property_type', '__indexed_occupancy_status', '__indexed_loan_purpose', '__indexed_channel', '__indexed_super_conforming_flag', '__indexed_harp_indicator', '__indexed_property_state']

Output columns:
['__encoded_first_time_homebuyer_flag', '__encoded_property_type', '__encoded_occupancy_status', '__encoded_loan_purpose', '__encoded_channel', '__encoded_super_conforming_flag', '__encoded_harp_indicator', '__encoded_property_state']

Category sizes:
[4, 6, 4, 4, 5, 3, 3, 55]


In [16]:
numerical_features = [
    "number_of_borrowers",
    "mi_percentage",
    "original_upb",
    "original_interest_rate",
    "original_loan_term",
    "current_actual_upb",
    "current_interest_rate",
    "estimated_ltv",
    "calculated_loan_age",
    "remaining_months_to_legal_maturity",
    "current_dpd_numeric",
    "max_dpd_to_date",
    "ever_30dpd_to_date",
    "delinquency_months_to_date",
    "months_since_last_delinquency",
    "upb_change_from_origination",
    "upb_pct_change_from_origination",
]

categorical_features = {
    "first_time_homebuyer_flag": 4,
    "property_type": 6,
    "occupancy_status": 4,
    "loan_purpose": 4,
    "channel": 5,
    "super_conforming_flag": 3,
    "harp_indicator": 3,
    "property_state": 55,
}

feature_mapping = []

# Numerical
for feature in numerical_features:
    feature_mapping.append(feature)

# One-hot categorical
for feature, size in categorical_features.items():
    for i in range(size):
        feature_mapping.append(f"{feature}_{i}")

# Final engineered feature
feature_mapping.append("original_dti_missing")

mapping_df = pd.DataFrame(
    {
        "model_feature": [f"f{i}" for i in range(len(feature_mapping))],
        "feature_name": feature_mapping,
    }
)

mapping_df

,model_feature,feature_name
0,f0,number_of_borrowers
1,f1,mi_percentage
2,f2,original_upb
3,f3,original_interest_rate
4,f4,original_loan_term
...,...,...
97,f97,property_state_51
98,f98,property_state_52
99,f99,property_state_53
100,f100,property_state_54


In [17]:
importance_mapped = (
    importance_df.rename(columns={"feature": "model_feature"})
    .merge(mapping_df, on="model_feature", how="left")
    .sort_values("importance", ascending=False)
)

importance_mapped.head(20)

,model_feature,importance,importance_pct,feature_name
0,f10,3692.408447,26.866874,current_dpd_numeric
1,f11,2280.193604,16.591251,max_dpd_to_date
2,f6,1646.572632,11.980868,current_interest_rate
3,f8,844.274414,6.143149,calculated_loan_age
4,f47,493.572784,3.591357,property_state_1
5,f3,479.788757,3.491061,original_interest_rate
6,f26,332.900726,2.422268,property_type_5
7,f17,296.057343,2.154186,first_time_homebuyer_flag_0
8,f0,267.407776,1.945725,number_of_borrowers
9,f27,263.220490,1.915257,occupancy_status_0


In [18]:
import numpy as np


def get_parent_feature(name):
    for feature in categorical_features:
        if name.startswith(feature + "_"):
            return feature
    return name


importance_mapped["parent_feature"] = importance_mapped["feature_name"].apply(
    get_parent_feature
)

parent_importance = (
    importance_mapped.groupby("parent_feature", as_index=False)
    .agg(
        total_gain=("importance", "sum"),
        total_gain_pct=("importance_pct", "sum"),
        dimensions_used=("model_feature", "count"),
    )
    .sort_values("total_gain", ascending=False)
    .reset_index(drop=True)
)

parent_importance["rank"] = np.arange(1, len(parent_importance) + 1)

parent_importance[["rank", "parent_feature", "total_gain_pct", "dimensions_used"]]

,rank,parent_feature,total_gain_pct,dimensions_used
0,1,current_dpd_numeric,26.866874,1
1,2,max_dpd_to_date,16.591251,1
2,3,property_state,12.963923,39
3,4,current_interest_rate,11.980868,1
4,5,calculated_loan_age,6.143149,1
5,6,original_interest_rate,3.491061,1
6,7,property_type,3.099463,4
7,8,occupancy_status,2.754209,3
8,9,first_time_homebuyer_flag,2.349491,2
9,10,number_of_borrowers,1.945725,1


In [19]:
state_importance = (
    importance_mapped[importance_mapped["parent_feature"] == "property_state"][
        ["feature_name", "importance", "importance_pct"]
    ]
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

state_importance["cumulative_gain_pct"] = (
    state_importance["importance_pct"].cumsum()
    / state_importance["importance_pct"].sum()
    * 100
)

state_importance.head(15)

,feature_name,importance,importance_pct,cumulative_gain_pct
0,property_state_1,493.572784,3.591357,27.702705
1,property_state_0,175.324158,1.275702,37.543104
2,property_state_6,142.020569,1.033377,45.514277
3,property_state_54,108.219658,0.787433,51.588309
4,property_state_51,85.080147,0.619064,56.363593
5,property_state_12,81.337387,0.591831,60.928808
6,property_state_27,73.789062,0.536907,65.070358
7,property_state_10,64.367912,0.468357,68.683129
8,property_state_5,47.964294,0.349000,71.375215
9,property_state_7,42.475128,0.309059,73.759212


In [20]:
state_indexer = [
    s
    for s in preprocessor.stages
    if type(s).__name__ == "StringIndexerModel" and s.getInputCol() == "property_state"
][0]

state_labels = state_indexer.labels

for i, state in enumerate(state_labels):
    print(i, state)

0 CA
1 TX
2 FL
3 IL
4 MI
5 OH
6 CO
7 NY
8 WA
9 NC
10 GA
11 AZ
12 PA
13 NJ
14 VA
15 MN
16 MA
17 IN
18 MO
19 OR
20 WI
21 TN
22 MD
23 UT
24 SC
25 KY
26 NV
27 AL
28 LA
29 CT
30 KS
31 IA
32 OK
33 ID
34 AR
35 NH
36 NE
37 NM
38 MT
39 ME
40 HI
41 MS
42 DE
43 RI
44 WV
45 VT
46 ND
47 DC
48 AK
49 SD
50 WY
51 GU
52 PR
53 VI


In [21]:
state_importance["state"] = (
    state_importance["feature_name"]
    .str.extract(r"property_state_(\d+)")[0]
    .astype(int)
    .map(dict(enumerate(state_labels)))
)

state_importance[["state", "importance_pct"]].head(15)

,state,importance_pct
0,TX,3.591357
1,CA,1.275702
2,CO,1.033377
3,NaN,0.787433
4,GU,0.619064
5,PA,0.591831
6,AL,0.536907
7,GA,0.468357
8,OH,0.349000
9,NY,0.309059


In [22]:
delinq_features = [
    "current_dpd_numeric",
    "max_dpd_to_date",
    "ever_30dpd_to_date",
    "delinquency_months_to_date",
    "months_since_last_delinquency",
]

delinq_importance = parent_importance[
    parent_importance["parent_feature"].isin(delinq_features)
][["parent_feature", "total_gain_pct"]].sort_values("total_gain_pct", ascending=False)

delinq_importance

,parent_feature,total_gain_pct
0,current_dpd_numeric,26.866874
1,max_dpd_to_date,16.591251
10,delinquency_months_to_date,1.746301
13,ever_30dpd_to_date,1.406948
22,months_since_last_delinquency,0.234742


In [23]:
delinq_importance["total_gain_pct"].sum()

np.float64(46.84611632836063)

In [24]:
train_df = spark.read.parquet("C:/Users/vorad/OneDrive/Desktop/Projects/mortgage-credit-risk/data/04_model_split/behavioral/train_split.parquet")

In [25]:
delinq_features = [
    "current_dpd_numeric",
    "max_dpd_to_date",
    "ever_30dpd_to_date",
    "delinquency_months_to_date",
    "months_since_last_delinquency",
]

train_df.select(delinq_features).describe().show()

+-------+--------------------+--------------------+------------------+--------------------------+-----------------------------+
|summary| current_dpd_numeric|     max_dpd_to_date|ever_30dpd_to_date|delinquency_months_to_date|months_since_last_delinquency|
+-------+--------------------+--------------------+------------------+--------------------------+-----------------------------+
|  count|            10970334|            10970334|          10970334|                  10970334|                       233580|
|   mean|0.005639026122632182|0.022978516424385986|               0.0|       0.03363252203624794|            3.837113622741673|
| stddev| 0.08353997541073933| 0.16069724172200223|               0.0|       0.29306794251981516|            3.521187117363373|
|    min|                 0.0|                 0.0|                 0|                         0|                          0.0|
|    max|                 2.0|                 2.0|                 0|                        12|       

In [ ]:
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=delinq_features, outputCol="delinq_vector")

corr_df = assembler.transform(train_df.select(delinq_features).dropna())

corr_matrix = Correlation.corr(corr_df, "delinq_vector", method="pearson").head()[0]

print(corr_matrix.toArray())

[[ 1.          0.3929698          nan  0.40470588 -0.56695761]
 [ 0.3929698   1.                 nan  0.47220784 -0.1664263 ]
 [        nan         nan  1.                 nan         nan]
 [ 0.40470588  0.47220784         nan  1.         -0.31492252]
 [-0.56695761 -0.1664263          nan -0.31492252  1.        ]]


In [ ]:
train_df.select("ever_30dpd_to_date").distinct().show()

+------------------+
|ever_30dpd_to_date|
+------------------+
|                 0|
+------------------+



In [ ]:
train_df.select("months_since_last_delinquency").groupBy(
    "months_since_last_delinquency"
).count().orderBy("months_since_last_delinquency").show(20)

+-----------------------------+--------+
|months_since_last_delinquency|   count|
+-----------------------------+--------+
|                         NULL|10736754|
|                          0.0|   54338|
|                          1.0|   26496|
|                          2.0|   22195|
|                          3.0|   21070|
|                          4.0|   22901|
|                          5.0|   26122|
|                          6.0|    8184|
|                          7.0|    7828|
|                          8.0|    7804|
|                          9.0|    8734|
|                         10.0|   11997|
|                         11.0|   15877|
|                         12.0|      34|
+-----------------------------+--------+



In [26]:
imputer = preprocessor.stages[0]

print(imputer.getStrategy())
print(imputer.surrogateDF.select("months_since_last_delinquency").show())

median
+-----------------------------+
|months_since_last_delinquency|
+-----------------------------+
|                         -1.0|
+-----------------------------+

None


In [27]:
parent_importance.sort_values("total_gain_pct", ascending=False).head(20)

,parent_feature,total_gain,total_gain_pct,dimensions_used,rank
0,current_dpd_numeric,3692.408447,26.866874,1,1
1,max_dpd_to_date,2280.193604,16.591251,1,2
2,property_state,1781.677249,12.963923,39,3
3,current_interest_rate,1646.572632,11.980868,1,4
4,calculated_loan_age,844.274414,6.143149,1,5
5,original_interest_rate,479.788757,3.491061,1,6
6,property_type,425.970039,3.099463,4,7
7,occupancy_status,378.520596,2.754209,3,8
8,first_time_homebuyer_flag,322.898829,2.349491,2,9
9,number_of_borrowers,267.407776,1.945725,1,10


In [28]:
from pyspark.sql import functions as F

(
    train_df.groupBy(
        (F.floor(F.col("calculated_loan_age") / 12) * 12).alias("age_bucket")
    )
    .agg(
        F.count("*").alias("population"),
        F.sum(F.col("future_90dpd_12m")).alias("events"),
        F.avg(F.col("future_90dpd_12m")).alias("event_rate"),
    )
    .orderBy("age_bucket")
    .show(30)
)

+----------+----------+------+--------------------+
|age_bucket|population|events|          event_rate|
+----------+----------+------+--------------------+
|         0|   5618700| 24828|0.004418815740295798|
|        12|   5351634| 49063|0.009167854154450771|
+----------+----------+------+--------------------+



In [29]:
(
    train_df.groupBy("calculated_loan_age")
    .agg(
        F.count("*").alias("population"),
        F.sum("future_90dpd_12m").alias("events"),
        F.avg("future_90dpd_12m").alias("event_rate"),
    )
    .orderBy("calculated_loan_age")
    .show(60)
)

+-------------------+----------+------+--------------------+
|calculated_loan_age|population|events|          event_rate|
+-------------------+----------+------+--------------------+
|                  6|   5618700| 24828|0.004418815740295798|
|                 12|   5351634| 49063|0.009167854154450771|
+-------------------+----------+------+--------------------+



In [30]:
(
    train_df.groupBy("occupancy_status")
    .agg(
        F.count("*").alias("population"),
        F.sum("future_90dpd_12m").alias("events"),
        F.avg("future_90dpd_12m").alias("event_rate"),
    )
    .orderBy(F.desc("event_rate"))
    .show()
)

+----------------+----------+------+--------------------+
|occupancy_status|population|events|          event_rate|
+----------------+----------+------+--------------------+
|               P|   9640730| 67192|0.006969596700664784|
|               I|    928229|  5164|0.005563282336578581|
|               S|    401375|  1535|0.003824353783867954|
+----------------+----------+------+--------------------+



In [31]:
(
    train_df.groupBy("calculated_loan_age", "occupancy_status")
    .agg(
        F.count("*").alias("population"),
        F.sum("future_90dpd_12m").alias("events"),
        F.avg("future_90dpd_12m").alias("event_rate"),
    )
    .orderBy("calculated_loan_age", "occupancy_status")
    .show()
)

+-------------------+----------------+----------+------+--------------------+
|calculated_loan_age|occupancy_status|population|events|          event_rate|
+-------------------+----------------+----------+------+--------------------+
|                  6|               I|    474710|  1553|0.003271471003349...|
|                  6|               P|   4938635| 22811|0.004618887607608175|
|                  6|               S|    205355|   464|0.002259501838280...|
|                 12|               I|    453519|  3611|0.007962180195317066|
|                 12|               P|   4702095| 44381|0.009438558770080145|
|                 12|               S|    196020|  1071|0.005463728191000918|
+-------------------+----------------+----------+------+--------------------+



In [32]:
rate_cols = [
    "original_interest_rate",
    "current_interest_rate",
]

rate_corr = (
    train_df.select(rate_cols)
    .dropna()
    .stat.corr("original_interest_rate", "current_interest_rate")
)

print(rate_corr)

0.9999969952611631


In [33]:
(
    train_df.select(
        (F.col("current_interest_rate") - F.col("original_interest_rate")).alias(
            "rate_change"
        )
    )
    .summary()
    .show()
)

+-------+--------------------+
|summary|         rate_change|
+-------+--------------------+
|  count|            10970334|
|   mean| 4.48846862821132E-7|
| stddev|0.001414265377495...|
|    min|              -0.625|
|    25%|                 0.0|
|    50%|                 0.0|
|    75%|                 0.0|
|    max|                 4.5|
+-------+--------------------+



In [34]:
(
    train_df.select(
        (F.col("current_interest_rate") - F.col("original_interest_rate")).alias(
            "rate_change"
        )
    )
    .agg(
        F.count("*").alias("total"),
        F.sum(F.when(F.col("rate_change") != 0, 1).otherwise(0)).alias("changed"),
    )
    .withColumn("changed_pct", F.col("changed") / F.col("total") * 100)
    .show()
)

+--------+-------+--------------------+
|   total|changed|         changed_pct|
+--------+-------+--------------------+
|10970334|     41|3.737352025927378E-4|
+--------+-------+--------------------+



In [35]:
(
    train_df.groupBy(
        F.when(F.col("estimated_ltv") < 60, "<60%")
        .when(F.col("estimated_ltv") < 70, "60-70%")
        .when(F.col("estimated_ltv") < 80, "70-80%")
        .when(F.col("estimated_ltv") < 90, "80-90%")
        .when(F.col("estimated_ltv") < 100, "90-100%")
        .otherwise("100%+")
        .alias("ltv_bucket")
    )
    .agg(
        F.count("*").alias("population"),
        F.sum("future_90dpd_12m").alias("events"),
        F.avg("future_90dpd_12m").alias("event_rate"),
    )
    .orderBy("ltv_bucket")
    .show()
)

+----------+----------+------+--------------------+
|ltv_bucket|population|events|          event_rate|
+----------+----------+------+--------------------+
|     100%+|   4767211| 21820|0.004577099692042161|
|    60-70%|   1080689|  7210|0.006671669647789512|
|    70-80%|   1855087| 15022|0.008097733421667017|
|    80-90%|   1191151| 13836|0.011615655781676714|
|   90-100%|    565761|  8231|0.014548546117530195|
|      <60%|   1510435|  7772|0.005145537543820158|
+----------+----------+------+--------------------+



In [36]:
(
    train_df.filter(F.col("estimated_ltv") != 999)
    .groupBy(
        F.when(F.col("estimated_ltv") < 60, "<60")
        .when(F.col("estimated_ltv") < 70, "60-70")
        .when(F.col("estimated_ltv") < 80, "70-80")
        .when(F.col("estimated_ltv") < 90, "80-90")
        .when(F.col("estimated_ltv") < 100, "90-100")
        .otherwise("100+")
        .alias("ltv_bucket")
    )
    .agg(
        F.avg("mi_percentage").alias("avg_mi"),
        F.avg("future_90dpd_12m").alias("event_rate"),
        F.count("*").alias("population"),
    )
    .orderBy("ltv_bucket")
    .show()
)

+----------+-------------------+--------------------+----------+
|ltv_bucket|             avg_mi|          event_rate|population|
+----------+-------------------+--------------------+----------+
|      100+|  18.85075777601551|0.012666314212705965|     45396|
|     60-70| 0.4561913742066404|0.006671669647789512|   1080689|
|     70-80|  2.331376372105459|0.008097733421667017|   1855087|
|     80-90| 20.740649170424238|0.011615655781676714|   1191151|
|    90-100|  26.74753473639929|0.014548546117530195|    565761|
|       <60|0.41698517314548456|0.005145537543820158|   1510435|
+----------+-------------------+--------------------+----------+



In [37]:
train_df.filter(
    (F.col("estimated_ltv") >= 100) & (F.col("estimated_ltv") != 999)
).select(
    F.min("estimated_ltv").alias("min"),
    F.max("estimated_ltv").alias("max"),
    F.count("*").alias("count"),
).show()

+---+---+-----+
|min|max|count|
+---+---+-----+
|100|902|45396|
+---+---+-----+



In [38]:
(
    train_df.filter((F.col("estimated_ltv") >= 100) & (F.col("estimated_ltv") != 999))
    .groupBy("estimated_ltv")
    .agg(
        F.count("*").alias("population"),
        F.avg("future_90dpd_12m").alias("event_rate"),
    )
    .orderBy("estimated_ltv")
    .show(100)
)

+-------------+----------+--------------------+
|estimated_ltv|population|          event_rate|
+-------------+----------+--------------------+
|          100|      9049|0.012156039341363686|
|          101|      6143|0.011557870747191926|
|          102|      4434|0.011502029769959404|
|          103|      3220|0.013043478260869565|
|          104|      2452|0.009787928221859706|
|          105|      1941|0.008758371973209686|
|          106|      1515|0.011881188118811881|
|          107|      1277|0.014095536413469069|
|          108|      1141|0.020157756354075372|
|          109|       973|0.013360739979445015|
|          110|       816|0.020833333333333332|
|          111|       740|0.014864864864864866|
|          112|       620|0.024193548387096774|
|          113|       596|0.020134228187919462|
|          114|       574|0.010452961672473868|
|          115|       531|0.018832391713747645|
|          116|       400|                0.01|
|          117|       492|0.004065040650

In [39]:
(
    train_df.groupBy(
        F.when(F.col("mi_percentage") == 0, "0%")
        .when(F.col("mi_percentage") < 5, "0-5%")
        .when(F.col("mi_percentage") < 10, "5-10%")
        .when(F.col("mi_percentage") < 15, "10-15%")
        .when(F.col("mi_percentage") < 20, "15-20%")
        .when(F.col("mi_percentage") < 30, "20-30%")
        .otherwise("30%+")
        .alias("mi_bucket")
    )
    .agg(
        F.count("*").alias("population"),
        F.sum("future_90dpd_12m").alias("events"),
        F.avg("future_90dpd_12m").alias("event_rate"),
    )
    .orderBy("mi_bucket")
    .show()
)

+---------+----------+------+--------------------+
|mi_bucket|population|events|          event_rate|
+---------+----------+------+--------------------+
|       0%|   7926263| 42915|0.005414279087130972|
|     0-5%|         6|     0|                 0.0|
|   10-15%|    412239|  3124|0.007578128221735449|
|   15-20%|     64180|  1412|0.022000623247117483|
|   20-30%|   1163018| 12875|0.011070335970724442|
|     30%+|   1349333| 13307|0.009861909550866984|
|    5-10%|     55295|   258|0.004665882991228863|
+---------+----------+------+--------------------+



In [40]:
(
    train_df.groupBy("mi_percentage")
    .agg(
        F.count("*").alias("population"),
        F.avg("future_90dpd_12m").alias("event_rate"),
    )
    .orderBy(F.desc("population"))
    .show(30)
)

+-------------+----------+--------------------+
|mi_percentage|population|          event_rate|
+-------------+----------+--------------------+
|            0|   7926263|0.005414279087130972|
|           30|   1339615|0.009763999358024507|
|           25|   1156998|0.011035455549620656|
|           12|    412195|0.007578937153531702|
|            6|     55284|0.004666811373996093|
|           18|     38774| 0.02408830659720431|
|           16|     24494|0.019065893688250183|
|           35|      9403|0.023290439221525044|
|           20|      5859|0.017921146953405017|
|           17|       864| 0.01273148148148148|
|           40|       166|0.012048192771084338|
|           33|        62| 0.03225806451612903|
|           22|        43|0.023255813953488372|
|           15|        32|                 0.0|
|           27|        28|                 0.0|
|           23|        25|                0.04|
|           32|        22| 0.09090909090909091|
|           26|        17|              

In [41]:
(
    train_df.select(F.col("upb_pct_change_from_origination").alias("upb_change_pct"))
    .summary()
    .show()
)

+-------+--------------------+
|summary|      upb_change_pct|
+-------+--------------------+
|  count|            10970334|
|   mean|-0.02620013819847098|
| stddev| 0.05422664810306527|
|    min| -0.9999999764150943|
|    25%|-0.02395209580838...|
|    50%|-0.01683501742160...|
|    75%|-0.00986842105263...|
|    max|0.030479999999999927|
+-------+--------------------+



In [42]:
(
    train_df.groupBy(
        F.when(F.col("upb_pct_change_from_origination") < -50, "< -50%")
        .when(F.col("upb_pct_change_from_origination") < -25, "-50 to -25%")
        .when(F.col("upb_pct_change_from_origination") < 0, "-25 to 0%")
        .when(F.col("upb_pct_change_from_origination") == 0, "0%")
        .when(F.col("upb_pct_change_from_origination") < 25, "0-25%")
        .when(F.col("upb_pct_change_from_origination") < 50, "25-50%")
        .otherwise("50%+")
        .alias("upb_change_bucket")
    )
    .agg(
        F.count("*").alias("population"),
        F.avg("future_90dpd_12m").alias("event_rate"),
    )
    .orderBy("upb_change_bucket")
    .show()
)

+-----------------+----------+--------------------+
|upb_change_bucket|population|          event_rate|
+-----------------+----------+--------------------+
|        -25 to 0%|  10860175|0.006673281047496932|
|               0%|    110064|0.012856156418084025|
|            0-25%|        95|0.031578947368421054|
+-----------------+----------+--------------------+



In [43]:
(train_df.select("current_actual_upb").summary().show())

+-------+------------------+
|summary|current_actual_upb|
+-------+------------------+
|  count|          10970334|
|   mean|222641.54648214253|
| stddev|117154.28632437601|
|    min|              0.01|
|    25%|          133000.0|
|    50%|         200319.91|
|    75%|          295000.0|
|    max|         1299000.0|
+-------+------------------+



In [44]:
(
    train_df.groupBy(
        F.when(F.col("current_actual_upb") < 100000, "<100K")
        .when(F.col("current_actual_upb") < 150000, "100-150K")
        .when(F.col("current_actual_upb") < 200000, "150-200K")
        .when(F.col("current_actual_upb") < 300000, "200-300K")
        .when(F.col("current_actual_upb") < 500000, "300-500K")
        .otherwise("500K+")
        .alias("upb_bucket")
    )
    .agg(
        F.count("*").alias("population"),
        F.sum("future_90dpd_12m").alias("events"),
        F.avg("future_90dpd_12m").alias("event_rate"),
    )
    .orderBy("upb_bucket")
    .show()
)

+----------+----------+------+--------------------+
|upb_bucket|population|events|          event_rate|
+----------+----------+------+--------------------+
|  100-150K|   2005444| 13478|0.006720706237621195|
|  150-200K|   1995371| 12929|0.006479496795332797|
|  200-300K|   2879671| 18213|0.006324680840276...|
|  300-500K|   2368761| 17128|0.007230784363639895|
|     500K+|    256394|  1888|0.007363666856478701|
|     <100K|   1464693| 10255|0.007001467201659324|
+----------+----------+------+--------------------+



In [45]:
(train_df.select("remaining_months_to_legal_maturity").summary().show())

+-------+----------------------------------+
|summary|remaining_months_to_legal_maturity|
+-------+----------------------------------+
|  count|                          10970334|
|   mean|                312.62792108243923|
| stddev|                 72.06194945007063|
|    min|                                48|
|    25%|                               348|
|    50%|                               348|
|    75%|                               354|
|    max|                               583|
+-------+----------------------------------+



In [46]:
(
    train_df.groupBy(
        F.when(F.col("remaining_months_to_legal_maturity") < 120, "<120")
        .when(F.col("remaining_months_to_legal_maturity") < 240, "120-240")
        .when(F.col("remaining_months_to_legal_maturity") < 300, "240-300")
        .when(F.col("remaining_months_to_legal_maturity") < 330, "300-330")
        .when(F.col("remaining_months_to_legal_maturity") < 360, "330-360")
        .otherwise("360+")
        .alias("maturity_bucket")
    )
    .agg(
        F.count("*").alias("population"),
        F.sum("future_90dpd_12m").alias("events"),
        F.avg("future_90dpd_12m").alias("event_rate"),
    )
    .orderBy("maturity_bucket")
    .show()
)

+---------------+----------+------+--------------------+
|maturity_bucket|population|events|          event_rate|
+---------------+----------+------+--------------------+
|        120-240|   2364482|  8010|0.003387634162577681|
|        240-300|     72071|   336|0.004662069348281556|
|        300-330|      7478|    29|0.003878042257288045|
|        330-360|   8439175| 65308|0.007738671137877814|
|           360+|       103|    16|  0.1553398058252427|
|           <120|     87025|   192|0.002206262568227...|
+---------------+----------+------+--------------------+

